# Safety Case Study Extraction

Metadata-only extraction for solver-vs-judge difficulty case studies.

Important: this notebook intentionally drops prompt/content columns such as `harmful_prompt` and `original_prompt`. Outputs include IDs, sources, attack families, difficulty estimates, uncertainty estimates, bins, and score dictionaries only.

In [ ]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd

BASE = Path("K-Factor/results/safety_solver_judge_comparison")
OUT = Path("K-Factor/paper_artifacts/safety/case_studies")
OUT.mkdir(parents=True, exist_ok=True)

paired_path = BASE / "safety_k2_solver_harmmetric_k2_paired_items.csv"
attack_path = BASE / "safety_k2_attack_family_detail.csv"
bin_summary_path = BASE / "safety_k2_difficulty_bin_summary.csv"

DROP_TEXT_COLS = [
    "harmful_prompt", "original_prompt", "judge_prompt", "solver_prompt",
    "prompt", "chosen", "rejected",
]

def score_values(score_str):
    if pd.isna(score_str):
        return []
    d = ast.literal_eval(score_str)
    vals = []
    for v in d.values():
        if v is None:
            continue
        try:
            fv = float(v)
        except (TypeError, ValueError):
            continue
        if np.isnan(fv):
            continue
        vals.append(fv)
    return vals

def save_both(df, stem):
    csv_path = OUT / f"{stem}.csv"
    json_path = OUT / f"{stem}.json"
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2)
    print(csv_path)
    print(json_path)

In [ ]:
paired = pd.read_csv(paired_path)
attack_detail = pd.read_csv(attack_path)
bin_summary = pd.read_csv(bin_summary_path)

# Safety guard: remove content-bearing columns before any display or save.
paired = paired.drop(columns=[c for c in DROP_TEXT_COLS if c in paired.columns])
attack_detail = attack_detail.drop(columns=[c for c in DROP_TEXT_COLS if c in attack_detail.columns])

for col in ["solver_scores", "judge_scores"]:
    paired[f"{col}_n"] = paired[col].apply(lambda s: len(score_values(s)))
    paired[f"{col}_mean"] = paired[col].apply(lambda s: np.mean(score_values(s)) if score_values(s) else np.nan)
    paired[f"{col}_sd"] = paired[col].apply(lambda s: np.std(score_values(s), ddof=1) if len(score_values(s)) > 1 else np.nan)

paired["solver_difficulty_percentile"] = paired["solver_difficulty_mean"].rank(pct=True) * 100
paired["judge_difficulty_percentile"] = paired["judge_difficulty_centered"].rank(pct=True) * 100
paired["difficulty_bin"] = pd.qcut(
    paired["solver_difficulty_mean"], q=10, labels=range(1, 11), duplicates="drop"
).astype(int)

attack_cols = [
    "prompt_id", "attack_family", "solver_attack_difficulty_centered",
    "solver_attack_difficulty_centered_laplace_se", "solver_scores",
]
attack_safe = attack_detail[[c for c in attack_cols if c in attack_detail.columns]].copy()
attack_safe["attack_solver_score_mean"] = attack_safe["solver_scores"].apply(
    lambda s: np.mean(score_values(s)) if score_values(s) else np.nan
)
attack_records = (
    attack_safe
    .sort_values(["prompt_id", "attack_family"])
    .groupby("prompt_id")
    .apply(lambda g: g.drop(columns=["prompt_id"], errors="ignore").to_dict("records"))
    .rename("attack_family_details")
    .reset_index()
)
paired = paired.merge(attack_records, on="prompt_id", how="left")

safe_cols = [
    "prompt_id", "source", "n_attacks", "attack_families", "difficulty_bin",
    "solver_difficulty_mean", "solver_difficulty_laplace_se_mean",
    "solver_difficulty_sd_across_attacks", "solver_difficulty_min", "solver_difficulty_max",
    "solver_difficulty_range_low", "solver_difficulty_range_high",
    "judge_difficulty_centered", "judge_difficulty_centered_laplace_se",
    "solver_difficulty_percentile", "judge_difficulty_percentile",
    "solver_scores_mean", "solver_scores_sd", "solver_scores_n",
    "judge_scores_mean", "judge_scores_sd", "judge_scores_n",
    "solver_scores", "judge_scores", "attack_family_details",
]
paired_safe = paired[[c for c in safe_cols if c in paired.columns]].copy()

paired_safe.head()

In [ ]:
hi_solver = paired_safe["solver_difficulty_percentile"].quantile(0.75)
lo_solver = paired_safe["solver_difficulty_percentile"].quantile(0.25)
hi_judge = paired_safe["judge_difficulty_percentile"].quantile(0.75)
lo_judge = paired_safe["judge_difficulty_percentile"].quantile(0.25)

specs = {
    "hard_to_solve__hard_to_judge": (
        (paired_safe["solver_difficulty_percentile"] >= hi_solver)
        & (paired_safe["judge_difficulty_percentile"] >= hi_judge),
        lambda d: np.minimum(d["solver_difficulty_percentile"], d["judge_difficulty_percentile"]),
    ),
    "easy_to_solve__hard_to_judge": (
        (paired_safe["solver_difficulty_percentile"] <= lo_solver)
        & (paired_safe["judge_difficulty_percentile"] >= hi_judge),
        lambda d: np.minimum(100 - d["solver_difficulty_percentile"], d["judge_difficulty_percentile"]),
    ),
    "hard_to_solve__easy_to_judge": (
        (paired_safe["solver_difficulty_percentile"] >= hi_solver)
        & (paired_safe["judge_difficulty_percentile"] <= lo_judge),
        lambda d: np.minimum(d["solver_difficulty_percentile"], 100 - d["judge_difficulty_percentile"]),
    ),
}

case_frames = []
summary_rows = []
for label, (mask, rank_fn) in specs.items():
    candidates = paired_safe.loc[mask].copy()
    candidates["case_category"] = label
    candidates["case_rank_score"] = rank_fn(candidates)
    candidates = candidates.sort_values("case_rank_score", ascending=False)
    selected = candidates.head(10).copy()
    case_frames.append(selected)
    summary_rows.append({
        "case_category": label,
        "n_available": int(len(candidates)),
        "n_selected": int(len(selected)),
        "solver_percentile_mean": selected["solver_difficulty_percentile"].mean(),
        "judge_percentile_mean": selected["judge_difficulty_percentile"].mean(),
        "solver_score_mean": selected["solver_scores_mean"].mean(),
        "judge_score_mean": selected["judge_scores_mean"].mean(),
        "solver_attack_sd_mean": selected["solver_difficulty_sd_across_attacks"].mean(),
    })

case_studies = pd.concat(case_frames, ignore_index=True)
case_summary = pd.DataFrame(summary_rows)

save_both(case_studies, "safety_k2_case_studies_metadata_only")
save_both(case_summary, "safety_k2_case_study_summary")
case_summary

In [ ]:
bin_9_10 = paired_safe[paired_safe["difficulty_bin"].isin([9, 10])].copy()
bin_9_10["bin_rank_high_judge"] = bin_9_10.groupby("difficulty_bin")["judge_scores_mean"].rank(ascending=False, method="first")
bin_9_10["bin_rank_low_judge"] = bin_9_10.groupby("difficulty_bin")["judge_scores_mean"].rank(ascending=True, method="first")

bin_9_10_examples = bin_9_10[
    (bin_9_10["bin_rank_high_judge"] <= 8)
    | (bin_9_10["bin_rank_low_judge"] <= 8)
].sort_values(["difficulty_bin", "bin_rank_high_judge"])

bin_9_10_summary = bin_9_10.groupby("difficulty_bin").agg(
    n_prompts=("prompt_id", "size"),
    solver_difficulty_mean=("solver_difficulty_mean", "mean"),
    solver_score_mean=("solver_scores_mean", "mean"),
    judge_score_mean=("judge_scores_mean", "mean"),
    judge_score_sd_across_prompts=("judge_scores_mean", "std"),
    solver_attack_sd_mean=("solver_difficulty_sd_across_attacks", "mean"),
    high_judge_count=("judge_scores_mean", lambda s: int((s >= 0.75).sum())),
    low_judge_count=("judge_scores_mean", lambda s: int((s <= 0.50).sum())),
).reset_index()

save_both(bin_9_10, "safety_k2_bin9_bin10_all_metadata_only")
save_both(bin_9_10_examples, "safety_k2_bin9_bin10_examples_metadata_only")
save_both(bin_9_10_summary, "safety_k2_bin9_bin10_summary")
save_both(bin_summary, "safety_k2_difficulty_bin_summary")

bin_9_10_summary